Compare manotorch with Yana's **manopth** and Omid's **MANO**. Clone both into `./thirdparty` (git-ignored):

manopth: https://github.com/hassony2/manopth, SHA `4f1dcad1201ff1bfca6e065a85f0e3456e1aa32b`  
MANO: https://github.com/otaheri/MANO,  SHA `5869ab059c1bf31cc724f57eaf93e041135e8960`

Both read the MANO pickles with chumpy, so this notebook needs `chumpy` installed, and manopth's own `mano` package on the path (`pip install -e thirdparty/manopth`). manotorch itself needs neither.

In [ ]:
import os, sys

module_path = os.path.abspath(os.path.join('..'))
sys.path.insert(0, module_path)  # test this repository, not an installed manotorch

import torch
import numpy as np
import thirdparty.manopth.manopth.manolayer as YanaMano
import thirdparty.MANO.mano as OmidMano
import manotorch.manolayer as ThisMano

### test PCA mode

In [ ]:
batch_size = 10
ncomps = 15

seed = 23
np.random.seed(seed)
torch.manual_seed(seed)

random_shape = torch.rand(batch_size, 10)
# Generate random pose parameters, including 3 values for global axis-angle rotation
random_pose = torch.rand(batch_size, 3 + ncomps)

In [ ]:
yana_mano_layer = YanaMano.ManoLayer(
    center_idx=None,
    side="right",
    mano_root="../assets/mano/models/",
    use_pca=True,
    flat_hand_mean=True,
    ncomps=ncomps
)

# hand's vertices and joints. The unit set by Yana is millimeters.
V1, J1 = yana_mano_layer(random_pose, random_shape)
# change to meter,
V1 = V1 / 1000.0
J1 = J1 / 1000.0

this_mano_layer = ThisMano.ManoLayer(
    rot_mode="axisang",
    center_idx=None,
    side="right",
    mano_assets_root="../assets/mano",
    use_pca=True,
    flat_hand_mean=True,
    ncomps=ncomps
)

out = this_mano_layer(random_pose, random_shape)
V2, J2 = out.verts, out.joints

TIPS = [4, 8, 12, 16, 20]  # fingertips in the 21-joint order
MANO16 = [i for i in range(21) if i not in TIPS]

assert torch.allclose(V1, V2, atol=1e-6)  # less than 0.001 mm is acceptable
assert torch.allclose(J1[:, MANO16], J2[:, MANO16], atol=1e-6)

manopth samples the fingertips at vertices 745, 317, 444, 556 and 673. manotorch uses the vertices of smplx (744, 320, 443, 554 and 671), so only the five fingertip joints differ.

In [ ]:
assert torch.allclose(J1[:, TIPS], V2[:, [745, 317, 444, 556, 673]], atol=1e-6)

In [ ]:
omid_mano_model = OmidMano.load(model_path="../assets/mano/models",
                     is_rhand= True,
                     num_pca_comps=ncomps,
                     batch_size=batch_size,
                     flat_hand_mean=True)

output = omid_mano_model(
    global_orient=random_pose[:,:3],
    hand_pose=random_pose[:,3:],
    transl = None,
    betas=random_shape,
    return_verts=True,
    return_tips = True
)

V3, J3 = output.vertices, output.joints
assert torch.allclose(V2, V3, atol=1e-6) # less than 0.001 mm is acceptable

Omid's MANO appends the same fingertip vertices as manotorch after the 16 MANO joints; only the joint order differs.

In [ ]:
REORDER_IDX = [0, 13, 14, 15, 16, 1, 2, 3, 17, 4, 5, 6, 18, 10, 11, 12, 19, 7, 8, 9, 20]

assert torch.allclose(J2, J3[:, REORDER_IDX], atol=1e-6)  # less than 0.001 mm is acceptable